# Multi-API Display Example

This example demonstrates the new clean multi-API approach where:
- All endpoints are defined at module level (no pickling issues)
- Multiple display objects share the same infrastructure
- Display objects just reference endpoint paths

In [1]:
import sys
sys.path.append('..')

from syft_widget import TimeDisplay, CPUDisplay, SystemDashboard
from syft_widget import start_infrastructure, stop_infrastructure, get_all_endpoints

## 1. View Available Endpoints

All endpoints are pre-defined in the library:

In [2]:
# See what endpoints are available
endpoints = get_all_endpoints()
print(f"Available endpoints ({len(endpoints)}):")
for path in sorted(endpoints.keys()):
    print(f"  {path}")

Available endpoints (6):
  /network/status
  /system/cpu
  /system/disk
  /system/memory
  /time/current
  /time/uptime


## 2. Start the Shared Infrastructure

This starts a single ManagedWidget that handles the checkpoint → thread → SyftBox lifecycle:

In [3]:
# Start the infrastructure (only needs to be done once)
start_infrastructure()

Starting widget infrastructure with 6 endpoints:
  /network/status
  /system/cpu
  /system/disk
  /system/memory
  /time/current
  /time/uptime
SyftBox server is running at http://localhost:57605
SyftBox app ready at http://localhost:57605
Waiting 15 seconds before stopping thread server (10s for frontend + 5s buffer)...
SyftBox server already available, skipping thread server

Infrastructure started:
  Thread server port: 8001
  SyftBox port: 8002
Already connected to SyftBox, monitoring for disconnection


## 3. Create Display Objects

Display objects are lightweight - they just reference endpoints and render data:

In [4]:
# Time display showing current time and uptime
time_display = TimeDisplay()
time_display

In [5]:
# CPU monitor with visual gauge
cpu_display = CPUDisplay()
cpu_display

In [6]:
# Complete system dashboard
dashboard = SystemDashboard()
dashboard

## 4. Custom Display Objects

You can easily create your own display objects:

In [7]:
from syft_widget import APIDisplay

class NetworkMonitor(APIDisplay):
    """Custom network monitor"""
    
    def __init__(self):
        super().__init__(endpoints=["/network/status"])
    
    def render_content(self, data, server_type="checkpoint"):
        net = data.get("/network/status", {})
        status = "🟢 Connected" if net.get('connected') else "🔴 Disconnected"
        
        # Server type badge
        badge_color = {"checkpoint": "#6c757d", "thread": "#28a745", "syftbox": "#007bff"}.get(server_type, "#6c757d")
        server_label = {"checkpoint": "📁 Checkpoint", "thread": "🧵 Thread Server", "syftbox": "📦 SyftBox"}.get(server_type, server_type)
        
        return f"""
        <div style="font-family: -apple-system, sans-serif; padding: 20px; background: #e8f5e9; border-radius: 8px; position: relative;">
            <div style="position: absolute; top: 10px; right: 10px; background: {badge_color}; color: white; padding: 4px 8px; border-radius: 4px; font-size: 12px;">
                {server_label}
            </div>
            
            <h3 style="margin: 0 0 15px 0;">🌐 Network Monitor</h3>
            
            <div style="font-size: 18px; margin-bottom: 10px;">{status}</div>
            
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px; font-size: 14px;">
                <div><strong>Interface:</strong> {net.get('interface', '...')}</div>
                <div><strong>IP:</strong> {net.get('ip_address', '...')}</div>
                <div><strong>Download:</strong> {net.get('download_mbps', '...')} Mbps</div>
                <div><strong>Upload:</strong> {net.get('upload_mbps', '...')} Mbps</div>
                <div><strong>Latency:</strong> {net.get('latency_ms', '...')} ms</div>
            </div>
        </div>
        """
    
    def get_update_script(self):
        return """
        const net = currentData['/network/status'] || {};
        const status = net.connected ? "🟢 Connected" : "🔴 Disconnected";
        
        // Server type badge
        const badgeColors = {checkpoint: "#6c757d", thread: "#28a745", syftbox: "#007bff"};
        const serverLabels = {checkpoint: "📁 Checkpoint", thread: "🧵 Thread Server", syftbox: "📦 SyftBox"};
        const badgeColor = badgeColors[currentServerType] || "#6c757d";
        const serverLabel = serverLabels[currentServerType] || currentServerType;
        
        element.innerHTML = `
        <div style="font-family: -apple-system, sans-serif; padding: 20px; background: #e8f5e9; border-radius: 8px; position: relative;">
            <div style="position: absolute; top: 10px; right: 10px; background: ${badgeColor}; color: white; padding: 4px 8px; border-radius: 4px; font-size: 12px;">
                ${serverLabel}
            </div>
            
            <h3 style="margin: 0 0 15px 0;">🌐 Network Monitor</h3>
            
            <div style="font-size: 18px; margin-bottom: 10px;">${status}</div>
            
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px; font-size: 14px;">
                <div><strong>Interface:</strong> ${net.interface || '...'}</div>
                <div><strong>IP:</strong> ${net.ip_address || '...'}</div>
                <div><strong>Download:</strong> ${net.download_mbps || '...'} Mbps</div>
                <div><strong>Upload:</strong> ${net.upload_mbps || '...'} Mbps</div>
                <div><strong>Latency:</strong> ${net.latency_ms || '...'} ms</div>
            </div>
        </div>
        `;
        """

# Create and display
network = NetworkMonitor()
network

## 5. Multiple Displays Together

All displays share the same backend infrastructure:

In [8]:
from IPython.display import display
from ipywidgets import HBox, VBox, HTML

# Create multiple displays
displays = [
    TimeDisplay(),
    CPUDisplay(),
    NetworkMonitor()
]

# Show them all
for d in displays:
    display(d)

## 6. Understanding the Architecture

Key points:
- **One infrastructure, many displays**: Single ManagedWidget handles all lifecycle
- **Fixed endpoints**: All endpoints defined in `endpoints.py` (no pickling issues)
- **Lightweight displays**: Display objects just render data, no server management
- **Automatic updates**: JavaScript polls the active server and updates displays

The lifecycle works exactly as before:
1. **Checkpoint**: Shows mock data immediately
2. **Thread server**: Starts automatically, serves dynamic data
3. **SyftBox app**: Takes over when available

## 7. Clean Up

When done, stop the infrastructure:

In [9]:
# This stops the thread server and cleans up
stop_infrastructure()

Stopping ManagedWidget...
Stopping monitoring thread...
_stop_thread_server called. Current thread_server: None
No thread server to stop
Killing any remaining processes on port 8002
ManagedWidget stopped
Widget infrastructure stopped
